In [ ]:
using NPZ
using Ripserer
using Pickle
using ProgressMeter
using Statistics

In [ ]:
data_dir = joinpath(get(ENV, "TOPOFINGER_HCP_ROOT", "./data/HCP/1200subjects/schaefer300"), "NoGSR_LowPassOnly") * "/"
data_dir_ts = joinpath(data_dir, "ts")
outputdir = joinpath(data_dir, "scaffolds", "whole", "representatives")
mkpath(outputdir)

# Parameters
lr = "LR"
atlas = "Schaefer300"
gsr = "NoGSR"
bandpass = "LowPassOnly"

files = readdir(data_dir_ts)

# Extract unique subject IDs (first element of filename split by '_')
subjects = sort(collect(Set(split(f, '_')[1] for f in files)))

println("Number of subjects: ", length(subjects))

In [ ]:
println("Number of threads: ", Threads.nthreads())

In [ ]:
n = length(subjects)
p = ProgressMeter.Progress(n)

Threads.@threads for sub in subjects
    # Find matching files for both sessions
    sess_files = String[]
    valid_subject = true
    for n_sess in (1, 2)
        pattern_base = "$(sub)_rfMRI_REST$(n_sess)_$(lr)_$(atlas)+Tian_$(gsr)_$(bandpass)"
        found_idx = findfirst(f -> startswith(f, pattern_base), files)
        if found_idx === nothing
            valid_subject = false
            break
        end
        push!(sess_files, files[found_idx])
    end

    if !valid_subject
        ProgressMeter.next!(p)
        continue
    end

    for (n_sess, fname) in enumerate(sess_files)
        session_idx = n_sess - 1
        file_path = joinpath(data_dir_ts, fname)
        output_file = joinpath(outputdir, "$(sub)_s$(session_idx).pck")

        # Skip if output already exists
        if isfile(output_file)
            continue
        end

        # Load timeseries
        data = npzread(file_path)
        ts = data["timeseries"]'

        # Compute correlation matrix and distance matrix
        pms = cor(ts)
        dist_corrs = 1 .- pms
        dist_corrs = (dist_corrs .+ transpose(dist_corrs)) / 2

        # Run persistent homology
        diagram = ripserer(dist_corrs; reps=1, alg=:homology, verbose=false)

        # Extract 1-cycle representatives
        cycles = []
        for interval in diagram[2]
            cycle = vertices.(representative(interval))
            push!(cycles, cycle)
        end

        store(output_file, cycles)
    end

    ProgressMeter.next!(p)
end

In [ ]:
# Quick check: inspect one output file
test_file = joinpath(outputdir, "$(subjects[1])_s0.pck")
if isfile(test_file)
    cycles = load(test_file)
    println("Subject: ", subjects[1])
    println("Number of 1-cycles: ", length(cycles))
    println("First cycle (edges): ", cycles[1])
else
    println("Output not found — run the main loop first.")
end

In [ ]:
# Dry-run on a single subject to verify pipeline
sub = subjects[1]
pattern_base = "$(sub)_rfMRI_REST1_$(lr)_$(atlas)+Tian_$(gsr)_$(bandpass)"
found_idx = findfirst(f -> startswith(f, pattern_base), files)
fname = files[found_idx]
file_path = joinpath(data_dir_ts, fname)

data = npzread(file_path)
ts = data["timeseries"]'
pms = cor(ts)
dist_corrs = 1 .- pms
dist_corrs = (dist_corrs .+ transpose(dist_corrs)) / 2

diagram = ripserer(dist_corrs; reps=1, alg=:homology, verbose=true)

cycles = []
for interval in diagram[2]
    cycle = vertices.(representative(interval))
    push!(cycles, cycle)
end

println("Subject: ", sub, " | Timeseries shape: ", size(ts))
println("Number of 1-cycles: ", length(cycles))
println("First cycle: ", cycles[1])